# N-BEATS experiment - Walmart Store Sales Forecasting

Person B track (DL + Prophet).

**This revision: a faithful darts port of a stronger N-BEATS recipe.** The
history of this notebook: it began on `darts.models.NBEATSModel` (Kaggle
Public 4289.38 / Private 4524.03 - the worst model in the project), was then
rewritten onto Nixtla `neuralforecast` `NBEATSx` after presentation feedback
said darts was the wrong tool. Neither version broke ~3500 on WMAE. A rival
team's darts N-BEATS reached **~3368**, and the lecturer asked us to close
that gap. Their notebook made the decisive choices this version now adopts.

**What actually moved their number (ranked):**
1. **The training loss *is* WMAE.** darts `NBEATSModel(loss_fn=torch.nn.L1Loss())`
   is fit with a per-timestep `sample_weight` TimeSeries that is `5.0` on
   `IsHoliday` weeks and `1.0` otherwise - so the L1 objective becomes the
   holiday-weighted MAE the competition scores on. Our neuralforecast version
   optimized an unweighted quantile loss; its own intro predicted it "won't
   score below 3000" for exactly this reason (the "amplitude gap"). darts
   supports this natively via `fit(sample_weight=...)`; neuralforecast does
   not, which is why this port goes back to darts.
2. **log1p target transform** over a per-series MinMax `Scaler`. Weekly sales
   span orders of magnitude across departments; `log1p` makes a single global
   model behave. Inverse path at predict is `inverse_transform -> expm1 ->
   clip at 0`.
3. **Balanced sampling** (`max_samples_per_ts=10`) so high-volume series do
   not dominate the global fit, and an **Optuna TPE** architecture search
   instead of a hand-picked 3-config comparison.

**Deliberately dropped: covariates.** The rival used *no* covariates and still
beat every previous N-BEATS by ~1150 WMAE, which says the loss + transform are
the real levers here, not features. darts' generic `NBEATSModel` is
target-only anyway (no future-covariate access), so this also sidesteps the
old past-covariates-only phase-shift bug. If the ported model still trails
~3500, the next lever is DLinear-style seasonal-anchor deseasonalization, not
covariates.

**The scaler discipline that matters.** darts pairs per-series scalers to the
input list *positionally*. The original darts run scored ~27,000 WMAE because
it fit the `Scaler` on the full series list but inverse-transformed a filtered
subset. Every stage below filters series to a usable set FIRST, then
fit/transform/inverse the `Scaler` on that one identical list.

**MLflow run plan** (under `NBEATS_Training`, `library="darts"`):
`NBEATS_Feature_Selection`, one `trial_XXX` run per Optuna trial, a
`NBEATS_CV_Best` summary, and `NBEATS_Final`.

##  Init cell (Colab-compatible)

Byte-identical to the TFT notebook's init cell.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    REPO_URL = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"
    REPO_DIR = "/content/walmart-sales-forecasting"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         f"{REPO_DIR}/requirements.txt", "--quiet"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "darts[torch]>=0.30", "optuna", "--quiet"], check=True)

    os.chdir(f"{REPO_DIR}/notebooks")

    from google.colab import drive
    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    repo_data_dir = Path(REPO_DIR) / "data" / "raw"
    if drive_data_dir.exists():
        subprocess.run(["cp", "-r", f"{drive_data_dir}/.", str(repo_data_dir)], check=True)
    else:
        raise FileNotFoundError(
            f"Expected Drive data folder not found at {drive_data_dir}. "
            "Create it (or add it as a My Drive shortcut) before running this notebook."
        )

sys.path.append(str(Path.cwd().parent))


Mounted at /content/drive


##  Imports

In [2]:
import json
import pickle
import time
import warnings
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import optuna

import torch
from darts import TimeSeries
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler

from src.preprocessing import load_raw_data, WalmartPreprocessor
from src.features import add_temporal_features
from src.evaluation import weighted_mae
from src.validation import expanding_window_splits, describe_split, split_series
from src.seasonal_anchor import smoothed_seasonal_anchor

pd.set_option("display.max_columns", 50)
pd.set_option("future.no_silent_downcasting", True)
torch.set_float32_matmul_precision("medium")
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=UserWarning)


###  Manual credentials override (VS Code / non-Colab-UI runtimes)

`google.colab.userdata` (Colab Secrets) can only be read from the Colab
**browser UI**. When the Colab runtime is driven from VS Code or any other
non-UI frontend it times out (`Secrets can only be fetched when running from
the Colab UI`). This cell sets the DagsHub creds directly instead, and the
credentials cell below skips `userdata` whenever these env vars are already
set.

`getpass` is used so the token is never written into this committed notebook -
run the cell and paste the values when prompted. Leave a prompt blank to fall
through to Colab Secrets / `.env` below (e.g. when you *are* on the Colab UI).


In [3]:
import os
from getpass import getpass

# Only prompt for values not already set, so re-running cells doesn't re-ask.
# Leave a prompt blank to fall through to Colab Secrets / .env in the next cell.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    _user = getpass("DagsHub username (blank -> use Colab Secrets / .env): ").strip()
    if _user:
        os.environ["MLFLOW_TRACKING_USERNAME"] = _user
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    _token = getpass("DagsHub token (blank -> use Colab Secrets / .env): ").strip()
    if _token:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _token


##  Load DagsHub credentials

`MLFLOW_TRACKING_USERNAME`/`MLFLOW_TRACKING_PASSWORD` are never hardcoded in
this notebook (it gets committed to the shared repo, so a hardcoded token
would leak).

- On the Colab UI: read from Colab secrets - add `DAGSHUB_USERNAME` and
  `DAGSHUB_TOKEN` via the key icon in the left sidebar, and enable
  "Notebook access" for both. Same approach as every other notebook.
- From VS Code / non-UI runtimes: use the manual-override cell above.
- Locally: falls back to a gitignored `.env` in the repo root.


In [4]:
if os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    # Already provided (e.g. by the manual-override cell above when driving the
    # Colab runtime from VS Code, where google.colab.userdata would time out).
    # Note: userdata.get(...) must NOT be evaluated in this case - it blocks for
    # ~minutes and raises when there is no Colab browser UI to answer it.
    creds_source = "pre-set environment variables"
elif IS_COLAB:
    from google.colab import userdata

    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
    creds_source = "Colab secrets (DAGSHUB_USERNAME / DAGSHUB_TOKEN)"
else:
    from dotenv import load_dotenv

    env_path = Path.cwd().parent / ".env"
    load_dotenv(env_path)
    creds_source = str(env_path)

assert os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"), (
    f"MLFLOW_TRACKING_USERNAME/PASSWORD not set (tried: {creds_source}). "
    "On the Colab UI: add DAGSHUB_USERNAME and DAGSHUB_TOKEN as Colab secrets "
    "(key icon in the left sidebar) and enable notebook access for both. "
    "From VS Code / non-UI runtimes: run the manual-override cell above. "
    "Locally: create a .env in the repo root with MLFLOW_TRACKING_USERNAME=... "
    "and MLFLOW_TRACKING_PASSWORD=..."
)
print("MLflow credentials loaded from:", creds_source)


MLflow credentials loaded from: pre-set environment variables


##  MLflow tracking store

Shared DagsHub MLflow server - the single source of truth for cross-model
WMAE comparison and the Model Registry, so all models log here rather than to
a per-notebook local store. Same experiment name as the previous Darts-based
runs (`NBEATS_Training`) so history stays comparable; each run below logs
`library="neuralforecast"` as a param to distinguish new runs from old ones.
Does not silently fall back to a local store if auth fails - that would
desync N-BEATS's runs from the rest of the team's.


In [5]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment("NBEATS_Training")
    mlflow.MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and "
        "MLFLOW_TRACKING_PASSWORD (a DagsHub access token) as environment "
        "variables, then re-run this cell. Not falling back to local "
        "./mlruns or sqlite - that would desync from the rest of the team's runs."
    ) from e

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("NBEATS_Training").name)


MLflow tracking URI: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow
Active experiment: NBEATS_Training


##  Load, merge, clean

Reuses `load_raw_data` / `WalmartPreprocessor` rather than reimplementing
loading logic. `WalmartPreprocessor.fit()` loads and caches `features_` /
`stores_` internally - `stores_` (Store, Type, Size) is what the
static-covariate section below builds on.


In [6]:
raw_data = load_raw_data(data_dir="../data/raw")

train_raw = raw_data["train"]
test_raw = raw_data["test"]

preprocessor = WalmartPreprocessor(data_dir="../data/raw")
preprocessor.fit(train_raw)

train_clean = preprocessor.transform(train_raw)
test_clean = preprocessor.transform(test_raw)

train_feat = add_temporal_features(train_clean)
test_feat = add_temporal_features(test_clean)

print(train_feat.shape, test_feat.shape)
train_feat.head()


(421570, 23) (115064, 22)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,week_of_year,month,year,days_to_super_bowl,days_to_labor_day,days_to_thanksgiving,days_to_christmas
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,5,2,2010,7,217,294,329
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315,6,2,2010,0,210,287,322
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315,7,2,2010,-7,203,280,315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315,8,2,2010,-14,196,273,308
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315,9,3,2010,-21,189,266,301


##  Model constants and compute budget

Fixed `input_chunk_length=52` (one seasonal cycle) and `output_chunk_length=13`;
the 39-week submission horizon is reached by darts rolling the 13-step model
forward autoregressively, matching the rival. `HOLIDAY_WEIGHT=5.0` is the value
that turns the L1 loss into WMAE. Hard `max_time` wall-clock ceilings guard
against a mis-attached (CPU) runtime silently blowing the budget.

In [7]:
FREQ = "W-FRI"
inferred_freq = pd.infer_freq(sorted(train_feat["Date"].unique()))
print("Inferred frequency from data:", inferred_freq)
assert inferred_freq in ("W-FRI", None), f"Unexpected frequency: {inferred_freq}"

INPUT_CHUNK_LENGTH = 52    # one year of history
OUTPUT_CHUNK_LENGTH = 13   # rolled autoregressively to HORIZON at submission time
HORIZON = 39               # full test horizon (~39 weeks past train end)
VAL_WEEKS = 13             # CV fold validation window - shared convention
MIN_REQUIRED_LENGTH = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH  # 65: min to form one sample

HOLIDAY_WEIGHT = 5.0       # 5x on holiday weeks -> L1 loss == WMAE
MAX_SAMPLES_PER_TS = 10    # balance the global fit across ~3,300 series
N_TRIALS = 20              # Optuna TPE trials (bump to 25 for a longer search)

BATCH_SIZE_SEARCH = 512
BATCH_SIZE_FINAL = 1024

GPU_AVAILABLE = torch.cuda.is_available()
print("=" * 60)
print("GPU available:", GPU_AVAILABLE)
print("=" * 60)

if IS_COLAB and not GPU_AVAILABLE:
    raise RuntimeError(
        "Running on Colab but no GPU is attached to THIS runtime session - "
        "training would silently fall back to CPU and blow the time budget. "
        "Runtime > Disconnect and delete runtime, then reconnect on a T4 GPU "
        "and Run all. Confirm this prints 'GPU available: True' first."
    )

# Wall-clock ceilings enforced via pl.Trainer max_time (per fit).
CV_MAX_TIME_MINUTES = 5     # per trial-fold fit (20 trials x 3 folds)
FINAL_MAX_TIME_MINUTES = 45 # single full-history fit


def make_trainer_kwargs(max_time_minutes=None, enable_progress_bar=False):
    """pl.Trainer kwargs forwarded through NBEATSModel(pl_trainer_kwargs=...),
    so accelerator/device/time-budget are consistent at every fit below."""
    kwargs = {
        "accelerator": "gpu" if GPU_AVAILABLE else "cpu",
        "enable_progress_bar": enable_progress_bar,
        "enable_model_summary": False,
    }
    if GPU_AVAILABLE:
        kwargs["devices"] = 1
    if max_time_minutes is not None:
        kwargs["max_time"] = timedelta(minutes=max_time_minutes)
    return kwargs


CEILING_MULTIPLIER = 1.5  # a forecast > 1.5x a series' own historical max is treated as a denorm blow-up


def clip_predictions(preds_df, pred_col, hist_max_by_key, global_max):
    """Clip to [0, CEILING_MULTIPLIER x that (Store, Dept)'s observed max].
    log1p+expm1 with a near-degenerate per-series MinMax range can still emit a
    runaway value; this bounds its damage to WMAE without touching normal rows.
    Returns (clipped_df, n_clipped)."""
    preds_df = preds_df.copy()
    keys = list(zip(preds_df["Store"], preds_df["Dept"]))
    ceilings = np.array([hist_max_by_key.get(k, global_max) for k in keys]) * CEILING_MULTIPLIER
    original = preds_df[pred_col].to_numpy()
    clipped = np.clip(original, 0.0, ceilings)
    n_clipped = int(np.sum(np.abs(clipped - original) > 1e-6))
    preds_df[pred_col] = clipped
    return preds_df, n_clipped


# Per-series historical max (raw sales) and global max - the clip ceilings.
hist_max_by_key = (
    train_feat.groupby(["Store", "Dept"])["Weekly_Sales"].max().to_dict()
)
GLOBAL_MAX = float(train_feat["Weekly_Sales"].max())

# Per (Store, Dept) raw history + dept/global means - the submission fallback.
history_by_key = {
    (s, d): g.set_index("Date")["Weekly_Sales"].sort_index()
    for (s, d), g in train_feat.groupby(["Store", "Dept"])
}
dept_mean = train_feat.groupby("Dept")["Weekly_Sales"].mean().to_dict()
GLOBAL_MEAN = float(train_feat["Weekly_Sales"].mean())
print(f"{len(history_by_key)} (Store, Dept) histories cached for fallback/clipping")


Inferred frequency from data: W-FRI
GPU available: True
3331 (Store, Dept) histories cached for fallback/clipping


##  Build the darts panel, holiday weights, and the log1p+scale transform

One target `TimeSeries` per `(Store, Dept)`. Every series is reindexed through
the **same** `TRAIN_END` so `predict(n=39)` yields the identical 39 future
dates (the test window) for every series - a discontinued department would
otherwise forecast off its own last week. Interior gaps are linearly
interpolated; the holiday sample-weight series is built on the same grid.

In [8]:
TRAIN_END = train_feat["Date"].max()


def build_darts_series(df, global_end, freq=FREQ):
    """One raw target TimeSeries per (Store, Dept), each reindexed from its own
    start through the shared `global_end`. Interior/trailing gaps linearly
    interpolated (limit_direction='both' holds the last value flat past the end).
    Returns (series_list, keys) in a single stable order."""
    series_list, keys = [], []
    for (store, dept), g in df.groupby(["Store", "Dept"]):
        g = g.sort_values("Date")
        full_range = pd.date_range(g["Date"].min(), global_end, freq=freq)
        y = g.set_index("Date")["Weekly_Sales"].reindex(full_range)
        y = y.interpolate(method="linear", limit_direction="both")
        y.index.name = "Date"
        series_list.append(TimeSeries.from_series(y, freq=freq))
        keys.append((store, dept))
    return series_list, keys


def build_holiday_weights(df, keys, global_end, freq=FREQ):
    """Per-series sample-weight TimeSeries on the same grid as each target:
    HOLIDAY_WEIGHT on IsHoliday weeks, else 1.0. darts weights each target
    timestep in the L1 loss with these, making the objective WMAE."""
    hol_by_date = (
        df.drop_duplicates(subset=["Date"]).set_index("Date")["IsHoliday"]
    )
    weight_list = []
    for (store, dept) in keys:
        g = df[(df["Store"] == store) & (df["Dept"] == dept)]
        start = g["Date"].min()
        full_range = pd.date_range(start, global_end, freq=freq)
        is_hol = hol_by_date.reindex(full_range).fillna(False).astype(bool)
        w = np.where(is_hol.to_numpy(), HOLIDAY_WEIGHT, 1.0).astype(np.float32)
        weight_list.append(TimeSeries.from_times_and_values(full_range, w, freq=freq))
    return weight_list


def to_log(ts_list):
    """Raw sales -> log1p (negatives, i.e. returns, clipped at 0 first)."""
    return [
        TimeSeries.from_times_and_values(
            ts.time_index, np.log1p(np.clip(ts.values(), 0.0, None))
        )
        for ts in ts_list
    ]


def from_log(ts_list):
    """log1p space -> raw sales (expm1). Clipping to >= 0 happens after."""
    return [
        TimeSeries.from_times_and_values(ts.time_index, np.expm1(ts.values()))
        for ts in ts_list
    ]


def preds_to_df(preds_list, keys):
    """darts prediction TimeSeries list -> long (Store, Dept, Date, pred) frame."""
    rows = []
    for (store, dept), ts in zip(keys, preds_list):
        s = ts.to_series()
        rows.append(pd.DataFrame({
            "Store": store, "Dept": dept,
            "Date": s.index, "Weekly_Sales_Pred": s.to_numpy(),
        }))
    return pd.concat(rows, ignore_index=True)


target_series, series_keys = build_darts_series(train_feat, global_end=TRAIN_END)
holiday_weights = build_holiday_weights(train_feat, series_keys, global_end=TRAIN_END)

assert all(len(t) == len(w) for t, w in zip(target_series, holiday_weights)), \
    "target/weight length mismatch"
assert all(t.end_time() == TRAIN_END for t in target_series), \
    "not every series reindexed to the shared TRAIN_END"
print(f"Built {len(target_series)} target series (+ holiday weights), "
      f"all ending {TRAIN_END.date()}")


Built 3331 target series (+ holiday weights), all ending 2012-10-26


##  Log the feature/covariate decision (`NBEATS_Feature_Selection`)

Recorded for the README: this darts N-BEATS is **target-only** (no future/past/
static covariates), unlike the neuralforecast `NBEATSx` attempt. The rival's
result is the evidence that the metric-aligned loss plus the log1p transform
outweigh covariates on this data.

In [9]:
mlflow.set_experiment("NBEATS_Training")

with mlflow.start_run(run_name="NBEATS_Feature_Selection"):
    mlflow.set_tag("stage", "Feature_Selection")
    mlflow.set_tag("library", "darts")
    mlflow.log_params({
        "target_columns_used": json.dumps(["Store", "Dept", "Date", "Weekly_Sales"]),
        "future_covariates": "none",
        "past_covariates": "none",
        "static_covariates": "none",
        "target_transform": "log1p",
        "per_series_scaling": "MinMax (darts Scaler, per series)",
        "loss": "L1Loss with 5x IsHoliday sample_weight == WMAE",
        "clip_negatives": True,
        "fallback_predictor": "smoothed lag-52 seasonal anchor -> dept mean -> global mean",
        "rationale": ("darts generic NBEATSModel is target-only; the rival's "
                      "no-covariate recipe beat every prior N-BEATS, so the "
                      "loss weighting + log1p transform are the levers, not features"),
    })
print("feature selection logged")


🏃 View run NBEATS_Feature_Selection at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/568dd91ac0084e81be5ab578b9ea51b2
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2
feature selection logged


##  Stage: `NBEATS_CV` - Optuna TPE hyperparameter search

`N_TRIALS` TPE trials x 3 expanding-window folds. Each fold fits a fresh darts
`NBEATSModel` on that fold's own training series (log1p -> per-series MinMax,
fit/transform/inverse on one identical filtered list), forecasts the 13-week
val window, inverse-transforms, clips, and scores `weighted_mae` against the
held-out actuals. Optuna minimizes the 3-fold mean WMAE. Every trial is logged
to MLflow as `trial_XXX`.

In [10]:
N_SPLITS = 3
MIN_TRAIN_WEEKS = 52
splits = expanding_window_splits(
    train_feat["Date"], n_splits=N_SPLITS, val_weeks=VAL_WEEKS, min_train_weeks=MIN_TRAIN_WEEKS
)
assert len(splits) == N_SPLITS, "history too short for the requested number of folds"
for i, split in enumerate(splits):
    print(f"fold {i}:", describe_split(split))


def build_fold(split):
    """RAW train/val target lists for one fold, filtered to series with enough
    training history AND a full val window, plus the train-portion holiday
    weights - all in one identical order (scaler-alignment safe)."""
    tr_list, va_list, w_list, fold_keys = [], [], [], []
    for ts, w, key in zip(target_series, holiday_weights, series_keys):
        try:
            tr, va = split_series(ts, split)
        except Exception:
            continue
        if len(tr) >= MIN_REQUIRED_LENGTH and len(va) == VAL_WEEKS:
            w_tr, _ = split_series(w, split)
            tr_list.append(tr); va_list.append(va); w_list.append(w_tr); fold_keys.append(key)
    return tr_list, va_list, w_list, fold_keys


# Precompute the (identical across trials) per-fold series sets once.
fold_cache = [build_fold(s) for s in splits]
for i, (tr, va, w, k) in enumerate(fold_cache):
    print(f"fold {i}: {len(k)} / {len(target_series)} series usable")

actual_df = train_feat[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]]


def make_model(params, max_time_minutes, batch_size):
    return NBEATSModel(
        input_chunk_length=INPUT_CHUNK_LENGTH,
        output_chunk_length=OUTPUT_CHUNK_LENGTH,
        generic_architecture=True,
        num_stacks=params["num_stacks"],
        num_blocks=params["num_blocks"],
        num_layers=params["num_layers"],
        layer_widths=params["layer_widths"],
        expansion_coefficient_dim=5,
        dropout=params["dropout"],
        n_epochs=params["n_epochs"],
        batch_size=batch_size,
        optimizer_kwargs={"lr": params["lr"]},
        loss_fn=torch.nn.L1Loss(),
        random_state=42,
        pl_trainer_kwargs=make_trainer_kwargs(max_time_minutes=max_time_minutes),
    )


def fit_predict_fold(params, fold):
    """Fit on one fold's train series (holiday-weighted L1, log1p+MinMax),
    predict the 13-week val window, return its WMAE in raw sales space."""
    tr_list, va_list, w_list, fold_keys = fold
    scaler = Scaler()
    tr_scaled = scaler.fit_transform(to_log(tr_list))   # filtered list only

    model = make_model(params, CV_MAX_TIME_MINUTES, BATCH_SIZE_SEARCH)
    model.fit(series=tr_scaled, sample_weight=w_list, max_samples_per_ts=MAX_SAMPLES_PER_TS)

    preds_scaled = model.predict(n=VAL_WEEKS, series=tr_scaled)
    preds_raw = from_log(scaler.inverse_transform(preds_scaled))   # same list length/order

    pred_df = preds_to_df(preds_raw, fold_keys)
    pred_df, _ = clip_predictions(pred_df, "Weekly_Sales_Pred", hist_max_by_key, GLOBAL_MAX)
    scored = pred_df.merge(actual_df, on=["Store", "Dept", "Date"], how="inner")
    return weighted_mae(scored["Weekly_Sales"], scored["Weekly_Sales_Pred"], scored["IsHoliday"])


def objective(trial):
    params = {
        "num_stacks": trial.suggest_categorical("num_stacks", [2, 4, 8]),
        "num_blocks": trial.suggest_categorical("num_blocks", [1, 2]),
        "num_layers": trial.suggest_categorical("num_layers", [2, 4]),
        "layer_widths": trial.suggest_categorical("layer_widths", [128, 256, 512]),
        "dropout": trial.suggest_categorical("dropout", [0.0, 0.1]),
        "lr": trial.suggest_categorical("lr", [1e-3, 3e-4]),
        "n_epochs": trial.suggest_categorical("n_epochs", [20, 25]),
    }
    fold_wmaes = [fit_predict_fold(params, fold) for fold in fold_cache]
    wmae_mean = float(np.mean(fold_wmaes))

    with mlflow.start_run(run_name=f"trial_{trial.number:03d}"):
        mlflow.set_tag("stage", "CV")
        mlflow.set_tag("library", "darts")
        mlflow.log_params({
            **params,
            "input_chunk_length": INPUT_CHUNK_LENGTH,
            "output_chunk_length": OUTPUT_CHUNK_LENGTH,
            "batch_size": BATCH_SIZE_SEARCH,
            "loss": "L1", "holiday_weight": HOLIDAY_WEIGHT,
            "target_transform": "log1p", "scaler": "per_series_minmax",
            "max_samples_per_ts": MAX_SAMPLES_PER_TS,
        })
        for i, w in enumerate(fold_wmaes):
            mlflow.log_metric(f"wmae_fold_{i}", w)
        mlflow.log_metric("wmae_mean", wmae_mean)
        mlflow.log_metric("wmae_std", float(np.std(fold_wmaes)))

    trial.set_user_attr("params_full", params)
    print(f"trial {trial.number:03d}: wmae_mean={wmae_mean:,.0f}")
    return wmae_mean


study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS)

BEST_PARAMS = study.best_trial.user_attrs["params_full"]
BEST_CV_WMAE = float(study.best_value)
print("\n" + "=" * 60)
print(f"best CV WMAE: {BEST_CV_WMAE:,.2f}")
print("best params:", BEST_PARAMS)
print("=" * 60)

with mlflow.start_run(run_name="NBEATS_CV_Best"):
    mlflow.set_tag("stage", "CV")
    mlflow.set_tag("library", "darts")
    mlflow.log_params({**BEST_PARAMS, "n_trials": N_TRIALS})
    mlflow.log_metric("best_cv_wmae", BEST_CV_WMAE)


fold 0: {'train_start': 'start_of_history', 'train_end': '2012-01-27 00:00:00', 'val_start': '2012-02-03 00:00:00', 'val_end': '2012-04-27 00:00:00', 'val_weeks': 13, 'n_holidays_in_val': 1}
fold 1: {'train_start': 'start_of_history', 'train_end': '2012-04-27 00:00:00', 'val_start': '2012-05-04 00:00:00', 'val_end': '2012-07-27 00:00:00', 'val_weeks': 13, 'n_holidays_in_val': 0}
fold 2: {'train_start': 'start_of_history', 'train_end': '2012-07-27 00:00:00', 'val_start': '2012-08-03 00:00:00', 'val_end': '2012-10-26 00:00:00', 'val_weeks': 13, 'n_holidays_in_val': 1}
fold 0: 3188 / 3331 series usable
fold 1: 3203 / 3331 series usable
fold 2: 3230 / 3331 series usable


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=25` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightnin

🏃 View run trial_000 at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/a4c0912798b246a4bd6a253c7f4cf9ac
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2
trial 000: wmae_mean=1,851


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightnin

🏃 View run trial_001 at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/48776e5061f444228794effedd11024e
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2
trial 001: wmae_mean=1,938


...

🏃 View run trial_017 at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/ef71c44df69e4cc0beb3aef76f35a31a
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2
trial 017: wmae_mean=1,893


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=25` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightnin

🏃 View run trial_018 at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/2721927ecdaf463fb84e0206298bd907
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2
trial 018: wmae_mean=1,887


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightnin

🏃 View run trial_019 at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/5a45d0416d324ec9bab63879b009516c
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2
trial 019: wmae_mean=1,816

best CV WMAE: 1,816.45
best params: {'num_stacks': 8, 'num_blocks': 1, 'num_layers': 4, 'layer_widths': 128, 'dropout': 0.0, 'lr': 0.001, 'n_epochs': 20}
🏃 View run NBEATS_CV_Best at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/6f99d69ad74348abbcbd2f16d538ccd1
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2


##  `.predict()` wrapper (`DartsNBeatsPipeline`) + final fit (`NBEATS_Final`)

The required raw-`test.csv`-in wrapper. It stores the fitted darts model, the
per-series `Scaler` (fit on the exact `fit_keys` list), those keys, and the
train history / fallback stats. `predict(raw_test_df)` preprocesses raw input,
re-runs the darts forecast, inverse-transforms (`inverse -> expm1 -> clip 0`),
maps each requested `(Store, Dept, Date)` to its forecast, and fills any series
with too little history via the smoothed lag-52 seasonal anchor (then dept mean,
then global mean).

In [11]:
class DartsNBeatsPipeline:
    """raw test.csv-shaped DataFrame in -> DataFrame[Id, Weekly_Sales] out.

    Wraps an already-fitted darts NBEATSModel trained on log1p+MinMax targets.
    Series too short to have been fit get a smoothed lag-52 seasonal-naive
    anchor (src.seasonal_anchor), falling back to dept mean then global mean."""

    def __init__(self, model, scaler, fit_series_scaled, fit_keys, preprocessor,
                 horizon, history_by_key, hist_max_by_key, global_max,
                 dept_mean, global_mean, ceiling_multiplier):
        self.model = model
        self.scaler = scaler
        self.fit_series_scaled = fit_series_scaled
        self.fit_keys = fit_keys
        self.preprocessor = preprocessor
        self.horizon = horizon
        self.history_by_key = history_by_key
        self.hist_max_by_key = hist_max_by_key
        self.global_max = global_max
        self.dept_mean = dept_mean
        self.global_mean = global_mean
        self.ceiling_multiplier = ceiling_multiplier

    def _forecast_lookup(self):
        # Inverse of the training transform, done inline so the pickled wrapper
        # carries no dependency on notebook-global helpers: MinMax inverse ->
        # expm1 -> clip 0.
        preds_log = self.scaler.inverse_transform(
            self.model.predict(n=self.horizon, series=self.fit_series_scaled)
        )
        lookup = {}
        for key, ts in zip(self.fit_keys, preds_log):
            vals = np.expm1(ts.values()).ravel()
            lookup[key] = pd.Series(vals, index=ts.time_index).clip(lower=0.0)
        return lookup

    def _fallback(self, store, dept, date):
        hist = self.history_by_key.get((store, dept))
        if hist is not None and len(hist):
            val = float(smoothed_seasonal_anchor(
                hist, [date], m=52, anchor_window=1, tolerance_days=3,
                fallback_level=float(hist.mean()),
            )[0])
            if np.isfinite(val):
                return max(val, 0.0)
        return max(self.dept_mean.get(dept, self.global_mean), 0.0)

    def predict(self, raw_test_df):
        test = add_temporal_features(self.preprocessor.transform(raw_test_df))
        lookup = self._forecast_lookup()

        out = []
        for r in test.itertuples(index=False):
            key = (r.Store, r.Dept)
            date = pd.Timestamp(r.Date)
            fc = lookup.get(key)
            if fc is not None and date in fc.index:
                val = float(fc.loc[date])
            else:
                val = self._fallback(r.Store, r.Dept, date)
            ceiling = self.hist_max_by_key.get(key, self.global_max) * self.ceiling_multiplier
            val = float(np.clip(val, 0.0, ceiling))
            out.append((f"{r.Store}_{r.Dept}_{date.date()}", val))
        return pd.DataFrame(out, columns=["Id", "Weekly_Sales"])


# ---- final fit on full history, best Optuna config ----
FINAL_PARAMS = {**BEST_PARAMS, "n_epochs": max(BEST_PARAMS["n_epochs"], 30)}

fit_mask = [len(ts) >= MIN_REQUIRED_LENGTH for ts in target_series]
fit_series = [ts for ts, m in zip(target_series, fit_mask) if m]
fit_weights = [w for w, m in zip(holiday_weights, fit_mask) if m]
fit_keys = [k for k, m in zip(series_keys, fit_mask) if m]
print(f"final fit on {len(fit_keys)} / {len(target_series)} series "
      f"(rest -> seasonal-anchor fallback)")

final_scaler = Scaler()
final_scaled = final_scaler.fit_transform(to_log(fit_series))   # one identical list

final_model = make_model(FINAL_PARAMS, FINAL_MAX_TIME_MINUTES, BATCH_SIZE_FINAL)
final_model.fit(series=final_scaled, sample_weight=fit_weights,
                max_samples_per_ts=MAX_SAMPLES_PER_TS)

pipeline = DartsNBeatsPipeline(
    model=final_model, scaler=final_scaler, fit_series_scaled=final_scaled,
    fit_keys=fit_keys, preprocessor=preprocessor, horizon=HORIZON,
    history_by_key=history_by_key, hist_max_by_key=hist_max_by_key,
    global_max=GLOBAL_MAX, dept_mean=dept_mean, global_mean=GLOBAL_MEAN,
    ceiling_multiplier=CEILING_MULTIPLIER,
)

with mlflow.start_run(run_name="NBEATS_Final") as final_run:
    final_run_id = final_run.info.run_id
    mlflow.set_tag("stage", "Final")
    mlflow.set_tag("library", "darts")
    mlflow.log_params({
        **FINAL_PARAMS,
        "input_chunk_length": INPUT_CHUNK_LENGTH,
        "output_chunk_length": OUTPUT_CHUNK_LENGTH,
        "horizon": HORIZON, "batch_size": BATCH_SIZE_FINAL,
        "loss": "L1", "holiday_weight": HOLIDAY_WEIGHT,
        "target_transform": "log1p", "scaler": "per_series_minmax",
        "max_samples_per_ts": MAX_SAMPLES_PER_TS,
        "n_fit_series": len(fit_keys),
    })
    mlflow.log_metric("best_cv_wmae", BEST_CV_WMAE)
    Path("../submissions").mkdir(exist_ok=True)
    # Pickle is for MLflow archival only - the submission cell uses the live
    # `pipeline` object, so a serialization hiccup on the darts model must not
    # block the run. darts' own model.save() is the portable alternative.
    try:
        with open("../submissions/_nbeats_pipeline.pkl", "wb") as f:
            pickle.dump(pipeline, f)
        mlflow.log_artifact("../submissions/_nbeats_pipeline.pkl")
        print("pipeline pickled and logged")
    except Exception as e:
        print("WARNING: could not pickle pipeline (submission still proceeds):", e)
        final_model.save("../submissions/_nbeats_model.pt")
        mlflow.log_artifact("../submissions/_nbeats_model.pt")
print("final model fit and logged to NBEATS_Final")


final fit on 3243 / 3331 series (rest -> seasonal-anchor fallback)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


pipeline pickled and logged
🏃 View run NBEATS_Final at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/db3bbe42cad34af7bf325f948691c168
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2
final model fit and logged to NBEATS_Final


##  Generate the submission CSV

Runs the wrapper on raw `test.csv`, reconciles against `sampleSubmission.csv`
(same Id set, same order), asserts no NaNs, and writes
`submissions/nbeats_submission.csv`.

In [12]:
submission = pipeline.predict(test_raw)

sample = pd.read_csv("../data/raw/sampleSubmission.csv")
submission = sample[["Id"]].merge(submission, on="Id", how="left")

assert submission["Weekly_Sales"].isna().sum() == 0, "submission has NaNs"
assert len(submission) == len(sample), f"row count {len(submission)} != {len(sample)}"
assert (submission["Id"].values == sample["Id"].values).all(), "Id order mismatch vs sample"

# Sanity: the forecast dates the model produced must cover the test window.
_test_dates = pd.to_datetime(pd.Series(test_raw["Date"]).unique())
_fc_dates = pipeline._forecast_lookup()[fit_keys[0]].index
assert set(_test_dates).issubset(set(_fc_dates)), \
    "model forecast dates do not cover the test window"

submission.to_csv("../submissions/nbeats_submission.csv", index=False)
print(f"wrote submissions/nbeats_submission.csv: {submission.shape[0]} rows")
print(f"Weekly_Sales: min={submission['Weekly_Sales'].min():.1f} "
      f"max={submission['Weekly_Sales'].max():.1f} "
      f"mean={submission['Weekly_Sales'].mean():.1f}")

with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact("../submissions/nbeats_submission.csv")
submission.head()


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try instal

wrote submissions/nbeats_submission.csv: 115064 rows
Weekly_Sales: min=-17.8 max=236191.2 mean=16076.7
🏃 View run NBEATS_Final at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/db3bbe42cad34af7bf325f948691c168
🧪 View experiment at: https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow/#/experiments/2


,Id,Weekly_Sales
0,1_1_2012-11-02,24159.055205
1,1_1_2012-11-09,25224.257184
2,1_1_2012-11-16,25568.525152
3,1_1_2012-11-23,25847.666627
4,1_1_2012-11-30,27928.565054
